# wmbench — Flexible — Kaggle benchmark

Runs **one attack at a time** with `--resume`, then zips `work/flexible/attacked/<attack>/` after each.

## Before you run
1. **Add Data**: `images500`, `negative500` (and flexible/ssl checkpoints if needed).
2. **Secrets**: `HF_TOKEN` (Hugging Face write token) for downloads.
3. Edit **CONFIG** cell: dataset paths, `HF_REPO_ID`, git clone URL.
4. **GPU**: T4 x2 or P100 recommended for Regen attacks; use `DIFF_BATCH=1` if OOM.

## Download results
Kaggle's Output download button is unreliable. This notebook uploads each zip to **Hugging Face** — use the printed `https://huggingface.co/datasets/...` links.

Alternatively: **Save Version** on this notebook, then on your PC:
`kaggle kernels output USER/THIS_NOTEBOOK -p ./out`


In [ ]:
# === CONFIG (edit paths) ===
METHOD = "flexible"
WAVES_ROOT = "/kaggle/working/waves"          # clone target
IMAGES = "/kaggle/input/images500/images"     # host images (not needed if GENERATE_BASED)
NEGATIVES = "/kaggle/input/negative500"       # clean negatives
OUTPUT_DIR = f"/kaggle/working/wmbench_results/{METHOD}"
EXPORT_DIR = f"/kaggle/working/wmbench_exports/{METHOD}"

# Attacks: run GPU set, DIST set, or ALL
RUN_GPU_ATTACKS = True
RUN_DIST_ATTACKS = True
SKIP_RINSE4X = True

# Download: upload each zip to Hugging Face (set HF_TOKEN in Kaggle Secrets)
UPLOAD_TO_HF = True
HF_REPO_ID = "YOUR_HF_USERNAME/wmbench-exports"  # dataset repo

GENERATE_BASED = False
DIFF_BATCH = 4
LPIPS_BATCH = 16
GENERATED_COUNT = 64   # tree-ring / generate-based only


In [ ]:
import os, sys, subprocess
from pathlib import Path

# Clone waves repo (use your fork if private)
WAVES_ROOT = Path("/kaggle/working/waves")
if not (WAVES_ROOT / "wmbench" / "run_benchmark.py").exists():
    # Option A: git clone (edit YOUR_USER). Option B: Add Data -> upload waves zip/dataset
    !git clone --depth 1 https://github.com/YOUR_USER/waves.git /kaggle/working/waves

os.chdir(WAVES_ROOT)
print("WAVES_ROOT:", WAVES_ROOT.resolve())

# PyTorch CUDA (adjust cu124 if needed)
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu124
!pip install -q -r wmbench/requirements_colab_wmbench_flexible.txt
!pip install -q git+https://github.com/openai/CLIP.git huggingface_hub

sys.path.insert(0, str(WAVES_ROOT))

# Hugging Face token from Kaggle Secrets (Add-ons -> Secrets -> HF_TOKEN)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ.setdefault("HF_TOKEN", UserSecretsClient().get_secret("HF_TOKEN"))
except Exception:
    pass


In [ ]:
# Flexible checkpoints (attach Kaggle dataset or unzip under /kaggle/working)
import os
from pathlib import Path

FLEX_ROOT = Path(os.environ.get("WMBENCH_FLEXIBLE_ROOT", "/kaggle/working/moe_stabilized_v2_milder_epoch200"))
os.environ["WMBENCH_FLEXIBLE_ROOT"] = str(FLEX_ROOT)
os.environ["WMBENCH_FLEXIBLE_CHECKPOINTS"] = str(FLEX_ROOT / "checkpoints")
os.environ["WMBENCH_FLEXIBLE_PROMPT"] = os.environ.get("WMBENCH_FLEXIBLE_PROMPT", "a photo")
print("FLEX_ROOT:", FLEX_ROOT, "exists:", FLEX_ROOT.exists())


In [ ]:
import os
import sys
from pathlib import Path

# Load config from previous cell
assert "METHOD" in dir(), "Run the config cell first."
WAVES_ROOT = Path(WAVES_ROOT)
sys.path.insert(0, str(WAVES_ROOT))

def _load_kaggle_common(waves_root):
    """Import kaggle_wmbench_common.py (works even if wmbench.notebooks is not a package)."""
    import importlib.util
    p = Path(waves_root) / "wmbench" / "notebooks" / "kaggle_wmbench_common.py"
    if not p.is_file():
        raise FileNotFoundError(
            f"Missing {p}. Upload the full waves repo (must include wmbench/notebooks/) "
            "or copy kaggle_wmbench_common.py into the notebook."
        )
    spec = importlib.util.spec_from_file_location("kaggle_wmbench_common", p)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

_common = _load_kaggle_common(WAVES_ROOT)
GPU_ATTACKS = _common.GPU_ATTACKS
DIST_ATTACKS = _common.DIST_ATTACKS
run_per_attack_with_zips = _common.run_per_attack_with_zips
zip_full_results = _common.zip_full_results
upload_to_huggingface = _common.upload_to_huggingface

attacks = []
if RUN_GPU_ATTACKS:
    attacks.extend(GPU_ATTACKS)
if RUN_DIST_ATTACKS:
    attacks.extend(DIST_ATTACKS)
if SKIP_RINSE4X:
    attacks = [a for a in attacks if a != "Rinse-4xDiff"]

out = Path(OUTPUT_DIR)
exp = Path(EXPORT_DIR)
exp.mkdir(parents=True, exist_ok=True)

images = None if GENERATE_BASED else IMAGES
negatives = None if GENERATE_BASED else NEGATIVES

extra = []
if GENERATE_BASED:
    extra = ["--generated-count", str(GENERATED_COUNT)]

entries = run_per_attack_with_zips(
    method=METHOD,
    waves_root=WAVES_ROOT,
    output_dir=out,
    attack_names=attacks,
    export_dir=exp,
    hf_repo_id=HF_REPO_ID if UPLOAD_TO_HF else None,
    upload_each=UPLOAD_TO_HF,
    images=images,
    negatives=negatives,
    generate_based=GENERATE_BASED,
    diff_batch=DIFF_BATCH,
    lpips_batch=LPIPS_BATCH,
    skip_rinse4x=SKIP_RINSE4X,
    extra_argv=extra,
)

print("\nDone per-attack zips:", len(entries))
for e in entries:
    print(e.get("hf_url") or e["zip"])


In [ ]:
# Final aggregate zip (CSVs + plots + full work/)
from pathlib import Path

_common = _load_kaggle_common(WAVES_ROOT)
full_zip = _common.zip_full_results(Path(OUTPUT_DIR), Path(EXPORT_DIR), METHOD)
print("Full results zip:", full_zip, f"({full_zip.stat().st_size/1e9:.3f} GB)")

if UPLOAD_TO_HF:
    url = _common.upload_to_huggingface(full_zip, HF_REPO_ID)
    print("Download full bundle:", url)

# List all exports in /kaggle/working (use HF links above, or Save Version -> Output tab)
for z in sorted(Path(EXPORT_DIR).glob("*.zip")):
    print(z, z.stat().st_size / 1e6, "MB")
